In [2]:
# 환경 설정
from inspect import FrameInfo
from google.colab import drive
drive.mount('/content/drive')

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np
import seaborn as sns
import requests
import xml.etree.ElementTree as ET
import pprint
import warnings
warnings.filterwarnings('ignore')

font_path = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/fonts/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
mpl.rc('font', family = 'NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# API 활용 데이터 수집
# https://data.seoul.go.kr/dataList/OA-15182/F/1/datasetView.do

api_key = '71474b655273796536324f6f616a61'
start = 1
end = 1000

In [17]:
# JSON
url = (f'http://openapi.seoul.go.kr:8088/{api_key}/json/tbCycleRentData/{start}/{end}/2026-03-30/1/')
data = requests.get(url, timeout=10).json() # JSON 형식으로 자동 파싱
# pprint.pprint(data)

data_dict = data.get('rentData', {})
print(f'전체 데이터 건수: {data_dict.get("list_total_count", "알 수 없음")}')

bike_rent_df = pd.DataFrame(data_dict.get('row', []))
bike_rent_df

전체 데이터 건수: 967


,BIKE_ID,RENT_DT,RENT_ID,RENT_NM,RENT_HOLD,RTN_DT,RTN_ID,RTN_NM,RTN_HOLD,USE_MIN,USE_DST,USR_CLS_CD,SEX_CD,BIRTH_YEAR,RENT_STATION_ID,RETURN_STATION_ID,BIKE_SE_CD,START_INDEX,END_INDEX,RNUM
0,SPB-40689,2026-03-30 01:00:13,02703,서울도시가스 앞,0,2026-03-30 01:01:32,01169,염창역 1번 출구,0,1,176.32,USR_001,M,2003,ST-2018,ST-1506,일반자전거,0,0,1
1,SPB-50048,2026-03-30 01:03:17,02085,강남중학교 앞,0,2026-03-30 01:03:50,02085,강남중학교 앞,0,0,0.00,USR_001,M,1995,ST-2358,ST-2358,일반자전거,0,0,2
2,SPB-47567,2026-03-30 01:00:36,00230,영등포구청역 7번출구,0,2026-03-30 01:05:08,03221,서울특별시 남부교육지원청,0,4,562.06,USR_001,M,2007,ST-413,ST-1992,일반자전거,0,0,3
3,SPB-52701,2026-03-30 01:00:10,01103,방화역 4번출구앞,0,2026-03-30 01:05:17,05053,신마곡벽산블루밍메트로오피스텔앞,0,5,776.02,USR_001,M,1969,ST-831,ST-2883,일반자전거,0,0,4
4,SPB-72070,2026-03-30 01:01:17,00213,KT앞,0,2026-03-30 01:05:20,00219,롯데캐슬엠파이어 옆,0,4,703.19,USR_001,M,1993,ST-59,ST-62,일반자전거,0,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
962,SPB-73495,2026-03-30 01:56:37,05865,더리브오피스텔 앞,0,2026-03-30 03:52:47,00222,시범아파트버스정류장 옆,0,116,4116.56,USR_001,M,1965,ST-3204,ST-68,일반자전거,0,0,963
963,SPB-63567,2026-03-30 01:01:40,00714,한국SGI 양천문화회관 앞,0,2026-03-30 04:26:04,NaN,NaN,NaN,20,3789.00,USR_001,F,1992,ST-321,NaN,일반자전거,0,0,964
964,SPB-68893,2026-03-30 01:18:18,00141,연대 대운동장 옆,0,2026-03-30 05:21:42,03009,홍대입구역 사거리,0,243,3025.73,USR_001,M,2003,ST-42,ST-2167,일반자전거,0,0,965
965,SPB-60972,2026-03-30 01:49:30,01337,돈암성당 옆,0,2026-03-30 08:34:39,00453,종로오가 지하쇼핑센터 14번출구,0,405,3088.75,USR_001,M,2008,ST-1201,ST-1606,일반자전거,0,0,966


In [18]:
bike_rent_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 967 entries, 0 to 966
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   BIKE_ID            967 non-null    object
 1   RENT_DT            967 non-null    object
 2   RENT_ID            967 non-null    object
 3   RENT_NM            967 non-null    object
 4   RENT_HOLD          967 non-null    object
 5   RTN_DT             967 non-null    object
 6   RTN_ID             962 non-null    object
 7   RTN_NM             962 non-null    object
 8   RTN_HOLD           962 non-null    object
 9   USE_MIN            967 non-null    object
 10  USE_DST            967 non-null    object
 11  USR_CLS_CD         967 non-null    object
 12  SEX_CD             773 non-null    object
 13  BIRTH_YEAR         938 non-null    object
 14  RENT_STATION_ID    967 non-null    object
 15  RETURN_STATION_ID  962 non-null    object
 16  BIKE_SE_CD         967 non-null    object
 1

In [20]:
BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'
DATA_PATH = BASE + 'seoul_bike_rent_history.csv'

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

bike_rent_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

In [21]:
# 컬럼명 매핑용 딕셔너리
bike_rent_columns = {
    'BIKE_ID': '자전거ID',
    'RENT_DT': '대여일시',
    'RENT_ID': '대여소번호',
    'RENT_NM': '대여소명',
    'RENT_HOLD': '대여거치대',
    'RTN_DT': '반납일시',
    'RTN_ID': '반납대여소번호',
    'RTN_NM': '반납대여소명',
    'RTN_HOLD': '반납거치대',
    'USE_MIN': '사용시간(분)',
    'USE_DST': '이용거리(M)',
    'USR_CLS_CD': '사용자종류',
    'SEX_CD': '성별',
    'BIRTH_YEAR': '생년',
    'RENT_STATION_ID': '대여대여소ID',
    'RETURN_STATION_ID': '반납대여소ID',
    'BIKE_SE_CD': '자전거구분'
}

---------------------------------------------------------------------------------------------------------------------

In [22]:
# API 활용 데이터 수집
# https://data.seoul.go.kr/dataList/OA-21235/S/1/datasetView.do

api_key = '706b77416573796534366a566d426a'
start = 1
end = 1000

In [23]:
# XML
url = (f'http://openapi.seoul.go.kr:8088/{api_key}/xml/bikeStationMaster/1/5/')
response = requests.get(url, timeout=10)
# HTTP 오류 발생 시 예외 처리
response.raise_for_status()

# XML 내용을 파싱합니다.
root = ET.fromstring(response.text)

# 전체 데이터 건수 추출
list_total_count_element = root.find('list_total_count')
list_total_count = list_total_count_element.text if list_total_count_element is not None else '알 수 없음'
print(f'전체 데이터 건수: {list_total_count}')

# 'row' 태그 아래의 데이터를 추출하여 DataFrame으로 변환
rows_data = []
for row_element in root.findall('row'):
    row_dict = {}
    for child in row_element:
        row_dict[child.tag] = child.text
    rows_data.append(row_dict)

bike_location_df = pd.DataFrame(rows_data)
bike_location_df

전체 데이터 건수: 3411


,RNTLS_ID,ADDR1,ADDR2,LAT,LOT
0,ST-10,서울특별시 마포구 양화로 93,427,37.552746,126.918617
1,ST-100,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.536667,127.073593
2,ST-1000,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.510380,126.866798
3,ST-1001,서울특별시 양천구 남부순환로4길20,서서울호수공원,0.000000,0.000000
4,ST-1002,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.529900,126.876541


In [24]:
bike_location_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   RNTLS_ID  5 non-null      object
 1   ADDR1     5 non-null      object
 2   ADDR2     5 non-null      object
 3   LAT       5 non-null      object
 4   LOT       5 non-null      object
dtypes: object(5)
memory usage: 332.0+ bytes


In [25]:
import os

BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'
DATA_PATH = BASE + 'seoul_bike_location_history.csv'

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

bike_location_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

In [26]:
# 컬럼명 매핑용 딕셔너리
bike_location_columns = {
    'RNTLS_ID': '대여소_ID',
    'ADDR1': '주소1',
    'ADDR2': '주소2',
    'LAT': '위도',
    'LOT': '경도'
}